<a href="https://colab.research.google.com/github/CooL-Legend/CSCI544-DLM/blob/adi/notebooks/sudoku/LLaDA_BGPO_Sudoku_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Original LLaDA vs BGPO: 4×4 Sudoku
Compare **GSAI-ML/LLaDA-8B-Instruct** (the original checkpoint before BGPO fine-tuning) and **THU-KEG/LLaDA-8B-BGPO-sudoku** on the authors' 500-puzzle test set. Both models receive the same original prompts and use the same baseline sampling settings.

Default: three puzzles, 256 output tokens, 256 steps, 32-token blocks, temperature 0, low-confidence remasking, seed 1. Start with `NUM_SAMPLES = 1` for a GPU smoke run. Increase to 3, 50, or 500 in Config Updater.

Choose an **L4 or A100** GPU and run cells in order. Each model runs in a fresh subprocess, sequentially, so only one 8B model occupies GPU memory. Three examples illustrate behavior; the full test set gives the broader accuracy comparison. Scores and differences below come from your actual runs.

The private repository checkout needs a fine-grained GitHub token with **Contents: read** access, stored as the Colab secret `GITHUB_TOKEN` or entered at the hidden prompt. New files must be available on `adi` before using the checkout cell.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select an L4 or A100 GPU runtime, then reconnect.")
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name} | GPU memory: {gpu.total_memory / 2**30:.1f} GiB")
if gpu.total_memory < 20 * 2**30:
    raise RuntimeError("This notebook uses unquantized BF16 weights. Choose an L4/A100 runtime with more GPU memory; a 16 GB T4 is too small for this setup.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("This GPU does not support the configured BF16 inference.")


## 1. Get the code from `adi`
Reuse the existing safe checkout workflow. Local changes and divergent commits stop the update without discarding work.

In [ ]:
from pathlib import Path
import getpass
import os
import subprocess
import sys
import tempfile

REPOSITORY = "CooL-Legend/CSCI544-DLM"
BRANCH = "adi"
REPO_URL = f"https://github.com/{REPOSITORY}.git"
REPO_DIR = Path("/content/CSCI544-DLM")

def run_git(args, token=None):
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    with tempfile.TemporaryDirectory() as tmp:
        if token:
            askpass = Path(tmp) / "askpass.py"
            askpass.write_text(
                "#!/usr/bin/env python3\nimport os, sys\n"
                "print('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['COLAB_GH_TOKEN'])\n"
            )
            askpass.chmod(0o700)
            env.update(GIT_ASKPASS=str(askpass), COLAB_GH_TOKEN=token)
        return subprocess.run(["git", "-c", "credential.helper=", *args], env=env, capture_output=True, text=True)

def remote_git(args):
    result = run_git(args)
    if result.returncode:
        from google.colab import userdata
        try:
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
        if not token:
            token = getpass.getpass("GitHub token with read access to this private repository: ")
        try:
            result = run_git(args, token)
        finally:
            del token
    if result.returncode:
        raise RuntimeError(result.stderr.strip() or "GitHub clone/update failed.")

def ensure_checkout():
    def local_git(*args):
        return subprocess.check_output(["git", "-C", str(REPO_DIR), *args], text=True).strip()

    if REPO_DIR.exists():
        if not (REPO_DIR / ".git").is_dir():
            raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout. Choose another REPO_DIR.")
        if local_git("remote", "get-url", "origin") != REPO_URL:
            raise RuntimeError("The existing checkout belongs to a different repository.")
        if local_git("status", "--porcelain"):
            raise RuntimeError("The checkout has local changes. Save them or choose another REPO_DIR before updating.")
        fetch_specs = local_git("config", "--get-all", "remote.origin.fetch").splitlines()
        branch_ref = f"refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}"
        if not any(spec.lstrip("+") in (branch_ref, "refs/heads/*:refs/remotes/origin/*") for spec in fetch_specs):
            local_git("remote", "set-branches", "--add", "origin", BRANCH)
        remote_git(["-C", str(REPO_DIR), "fetch", "origin", branch_ref])
        exists = subprocess.run(["git", "-C", str(REPO_DIR), "show-ref", "--verify", "--quiet", f"refs/heads/{BRANCH}"]).returncode == 0
        if exists:
            local_git("switch", BRANCH)
        else:
            local_git("switch", "--create", BRANCH, "--track", f"origin/{BRANCH}")
        local_git("merge", "--ff-only", f"origin/{BRANCH}")
    else:
        remote_git(["clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
    if local_git("branch", "--show-current") != BRANCH or local_git("rev-parse", "HEAD") != local_git("rev-parse", f"origin/{BRANCH}"):
        raise RuntimeError("The checkout differs from origin/adi. Review local commits before running experiments.")
    print(f"Code ready: {REPO_DIR} | branch: {BRANCH} | commit: {local_git('rev-parse', '--short', 'HEAD')}")

ensure_checkout()


## 2. Install comparison dependencies
Both inference subprocesses use Transformers **4.53.2**, matching BGPO's upstream requirements. Keep Colab's CUDA-enabled PyTorch. BGPO's loader explicitly disables the checkpoint's external FlashAttention flags and uses its PyTorch attention path. No FlashAttention compilation is needed.

Rerun this cell if switching from another model notebook in the same runtime.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements/bgpo-colab.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO_DIR)], check=True)
import site
for package_dir in [*site.getsitepackages(), site.getusersitepackages()]:
    site.addsitedir(package_dir)

## 3. Mount Drive
Allow at least **40 GB free Drive storage** for both model snapshots, plus results. An existing pinned LLaDA snapshot is reused. GPU memory holds one model at a time.

`COPY_MODEL_TO_RUNTIME = True` stages both models on runtime disk to speed repeated loads. Set it to `False` to load directly from Drive. Existing 9×9 results remain in their original folders.

In [ ]:
from google.colab import drive
import importlib

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/CSCI544-DLM")
COPY_MODEL_TO_RUNTIME = True
EXPERIMENT_TIMEZONE = "America/Los_Angeles"
from dlm_project.models import profiles as model_profiles
from dlm_project.experiments import paths as experiment_paths
importlib.reload(model_profiles)
importlib.reload(experiment_paths)
FAMILIES = {"baseline": "llada", "bgpo": "llada_bgpo"}
PROFILES = {name: model_profiles.get_profile(family) for name, family in FAMILIES.items()}
DRIVE_MODEL_DIRS = {name: DRIVE_ROOT / "models" / profile["repo_id"].split("/")[-1] for name, profile in PROFILES.items()}
RUNTIME_MODEL_DIRS = {name: Path("/content/models") / path.name for name, path in DRIVE_MODEL_DIRS.items()}
MODEL_PATHS = RUNTIME_MODEL_DIRS if COPY_MODEL_TO_RUNTIME else DRIVE_MODEL_DIRS
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["HF_HUB_CACHE"] = str(DRIVE_ROOT / "cache/hub")
os.environ["HF_DATASETS_CACHE"] = str(DRIVE_ROOT / "cache/datasets")
for name, path in DRIVE_MODEL_DIRS.items():
    print(name, "persistent model:", path)
print("Comparison root:", DRIVE_ROOT / "Comparisons/Sudoku4x4")

## 4. Load the shared configuration
The BGPO checkpoint is configured under `model`; the original LLaDA checkpoint is under `comparison.baseline_model`. Both share the same dataset and generation settings.

In [ ]:
import yaml

config = yaml.safe_load((REPO_DIR / "configs/sudoku/llada_bgpo_4x4.yaml").read_text())
config["data"]["cache_dir"] = os.environ["HF_DATASETS_CACHE"]
config["model"].update(model_path=str(MODEL_PATHS["bgpo"]), device="cuda", local_files_only=True)
config["comparison"]["baseline_model"].update(model_path=str(MODEL_PATHS["baseline"]), device="cuda", local_files_only=True)
config["experiment_root"] = str(DRIVE_ROOT)
config["experiment_timezone"] = EXPERIMENT_TIMEZONE
RUN_CONFIG = Path("/content/llada-bgpo-4x4-run.yaml")
RUNNER = REPO_DIR / "scripts/run_bgpo_sudoku.py"
for name, profile in PROFILES.items():
    print(name, profile["repo_id"], "revision:", profile["revision"])

## 5. Cache both models on Drive
The original LLaDA snapshot is reused when already present. BGPO has its own folder and pinned identity. Interrupted downloads and partial runtime copies can be retried by rerunning this cell. Keep the hidden Hugging Face cache metadata.

In [ ]:
for name, family in FAMILIES.items():
    command = [sys.executable, "-u", str(REPO_DIR / "scripts/cache_model.py"), "--model", family, "--drive-dir", str(DRIVE_MODEL_DIRS[name])]
    if COPY_MODEL_TO_RUNTIME:
        command += ["--runtime-dir", str(RUNTIME_MODEL_DIRS[name])]
    subprocess.run(command, check=True)

## Config Updater
Choose **1–500** test puzzles. Selection uses the first N rows of the pinned test set, without shuffling. The authors' original prompt asks for reasoning followed by a 16-digit solution inside `<answer>` tags. References are used only for scoring.

Both models use identical baseline settings. Seed `1 + puzzle index` is shared across the paired runs. Config Updater previews the actual first prompt without loading weights.

In [ ]:
NUM_SAMPLES = 3  # Use 1 first; then 3, 50, or the full 500.
config["data"]["num_samples"] = NUM_SAMPLES
config["generation"].update(gen_length=256, steps=256, block_length=32, temperature=0.0, remasking="low_confidence")
config["decoding"] = {"method": "baseline", "seed": 1}
from dlm_project.tasks.sudoku.model_comparison import comparison_configs
comparison_configs(config)
RUN_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
print(RUN_CONFIG.read_text())
print("Comparison parent:", DRIVE_ROOT / "Comparisons/Sudoku4x4" / experiment_paths.experiment_name(config["generation"], config["decoding"]))
subprocess.run([sys.executable, "-u", str(RUNNER), "--config", str(RUN_CONFIG), "--dataset-only"], check=True)

## 6. Run the paired experiment
The coordinator runs original LLaDA, waits for its process to exit, then runs BGPO. Each model saves predictions, raw responses, traces, metrics, and plots as it progresses. The comparison verifies matching puzzle IDs, prompts, checkpoint identities, and decoding settings before calculating differences.

Each invocation creates new timestamped directories. Failed or interrupted comparisons preserve completed individual artifacts; automatic resume is not implemented. `completed` means both evaluations and all plots finished, not that every puzzle was solved.

In [ ]:
import json
import uuid

latest = None
RUN_MANIFEST = Path("/content") / f"bgpo-comparison-{uuid.uuid4().hex}.json"
result = subprocess.run([sys.executable, "-u", str(RUNNER), "--config", str(RUN_CONFIG), "--run-manifest", str(RUN_MANIFEST)])
if RUN_MANIFEST.is_file():
    receipt = json.loads(RUN_MANIFEST.read_text())
    latest = Path(receipt["run_dir"])
    print("This comparison:", latest, "|", receipt["status"])
result.check_returncode()

## 7. Compare the measured results
The table contains exact and cell accuracy, completion, clue preservation, Sudoku validity, mean generation latency, and peak allocated GPU memory. Latency excludes model loading; memory includes resident model weights. Rate differences are **BGPO minus original LLaDA** in percentage points.

4×4 parsing requires one complete `<answer>` block containing exactly 16 digits from 1–4 after whitespace removal. Missing tags, truncated answers, and extra digits fail parsing and receive zero cell credit. Sudoku rules check rows, columns, and 2×2 boxes independently of clue preservation. No solver fills or repairs an answer.

In [ ]:
import pandas as pd
from IPython.display import display, Image

if latest is None:
    raise RuntimeError("Run the paired experiment first.")
summary = json.loads((latest / "comparison.json").read_text())
print("Comparison:", latest, "| status:", summary["status"])
print("Individual model runs:", summary["model_runs"])
if summary["status"] != "completed":
    raise RuntimeError("Comparison is incomplete; inspect the preserved individual runs and error above.")
table = pd.DataFrame(summary["models"]).set_index("model")
rates = ["exact_accuracy", "cell_accuracy", "completion_rate", "clue_preservation_rate", "sudoku_validity_rate"]
table["peak_gpu_memory_GiB"] = table.pop("peak_gpu_memory_bytes") / 2**30
display(table.style.format({**{key: "{:.1%}" for key in rates}, "mean_latency_seconds": "{:.2f}", "peak_gpu_memory_GiB": "{:.2f}"}))
display(pd.DataFrame([{"metric": key, "BGPO minus original (percentage points)": 100 * value} for key, value in summary["bgpo_minus_baseline"].items()]))
pairs = [json.loads(line) for line in (latest / "paired_predictions.jsonl").read_text().splitlines()]
print("Paired puzzles:", len(pairs))

### Inspect paired answers and four-panel plots
Show the first three examples by default. Change `PUZZLES_TO_DISPLAY` to inspect more saved results without reloading either model. Full raw responses and per-puzzle timing/memory statistics are retained in `paired_predictions.jsonl`.

In [ ]:
PUZZLES_TO_DISPLAY = 3
for pair in pairs[:PUZZLES_TO_DISPLAY]:
    print(f"\nTest puzzle {pair['source_index']} | puzzle: {pair['puzzle']} | reference: {pair['solution']}")
    rows = []
    for name in ("baseline", "bgpo"):
        record = pair[name]
        stats = record["decoding_stats"]
        print(f"{name} raw response:\n{record['raw_output']}\n")
        rows.append({"model": name, **{key: record[key] for key in ["prediction", "parse_success", "exact", "correct_cells", "clues_preserved", "valid_sudoku"]}, "latency_seconds": stats["latency_seconds"], "peak_gpu_memory_GiB": stats["peak_gpu_memory_bytes"] / 2**30})
    display(pd.DataFrame(rows))
    plot_path = latest / pair["plot_file"]
    display(Image(filename=str(plot_path)))

### Verify the actual CUDA run
Check both saved model identities, matching inputs, GPU usage, and completed plots. Passing these checks establishes successful execution; the measured scores describe answer quality.

In [ ]:
assert len(pairs) == config["data"]["num_samples"] == summary["plots_saved"]
for name, run_dir in summary["model_runs"].items():
    saved = json.loads((Path(run_dir) / "config.json").read_text())
    identity = saved["provenance"]["model"]
    assert identity["repo_id"] == PROFILES[name]["repo_id"]
    assert identity["revision"] == PROFILES[name]["revision"]
    if name == "bgpo":
        assert identity["attention_overrides"] == {"flash_attention": False, "flash_attn_varlen": False}
for pair in pairs:
    assert pair["baseline"]["prompt"] == pair["bgpo"]["prompt"]
    assert pair["baseline"]["source_index"] == pair["bgpo"]["source_index"]
    assert (latest / pair["plot_file"]).is_file()
    for name in ("baseline", "bgpo"):
        stats = pair[name]["decoding_stats"]
        assert torch.device(stats["model_device"]).type == "cuda"
        assert stats["peak_gpu_memory_bytes"] > 0
        assert stats["method"] == "baseline" and stats["search_model_calls"] == 0
        assert stats["completion_model_calls"] == config["generation"]["steps"]
print("Both checkpoints completed on CUDA; all paired plots are present.")

## 8. Check caching, then increase the sample count
After a one-puzzle smoke run, use three puzzles for the default comparison. Rerun step 5 and evaluation to verify that unchanged Drive/runtime model files are reused and the new results receive a new timestamp. Increase `NUM_SAMPLES` to 50 or all 500 when ready.

Artifacts:
- Per-model: `LLaDA/Sudoku4x4/<settings>/<timestamp>/` and `LLaDA-BGPO/Sudoku4x4/<settings>/<timestamp>/` contain configuration/provenance, metrics, predictions, traces, and plots.
- Comparison: `Comparisons/Sudoku4x4/<settings>/<timestamp>/` contains `comparison.json`, `comparison.csv`, `paired_predictions.jsonl`, child configurations/run receipts, and four-panel plots.

References:
- [BGPO checkpoint](https://huggingface.co/THU-KEG/LLaDA-8B-BGPO-sudoku)
- [Original LLaDA-8B-Instruct](https://huggingface.co/GSAI-ML/LLaDA-8B-Instruct)
- [Pinned authors' 4×4 test set](https://github.com/THU-KEG/BGPO/blob/2d3e819a65b42ff5bd9cce958fb4412d57c1609d/data/preprocessed/test/sudoku_1.parquet)
- [BGPO code](https://github.com/THU-KEG/BGPO)

This is a paired inference comparison on the released test set, using this repository's scoring rules. It does not reproduce the full training framework or claim the paper's reported scores.